In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected successfully!")

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nBASIC STATISTICS")
print(df.describe())


In [ ]:
print(df.isnull().sum())

In [ ]:
# Prepare the raw dataset for Snowflake

df_raw = df.copy()

# Convert the timestamp while preserving the +03:00 timezone
df_raw["time"] = pd.to_datetime(df_raw["time"])

print("Raw dataset ready:", df_raw.shape)
print(df_raw.dtypes)

In [ ]:
# Convert timestamps to timezone-aware UTC timestamps
df_raw["time"] = pd.to_datetime(df_raw["time"], utc=True)

print("Timestamp conversion successful!")
print("Time datatype:", df_raw["time"].dtype)
print("First timestamp:", df_raw["time"].iloc[0])

In [ ]:
# Load the raw Kaggle dataset into Snowflake

session.write_pandas(
    df_raw,
    table_name="RAW_ENERGY",
    database="ENERGY_ML",
    schema="ML_PIPE",
    auto_create_table=False,
    overwrite=True
)

print("RAW_ENERGY loaded successfully!")
print("Rows loaded:", len(df_raw))

In [ ]:
# Match pandas column names with the Snowflake table columns
df_raw.columns = [col.upper() for col in df_raw.columns]

print("Column names updated:")
print(df_raw.columns.tolist())

In [ ]:
session.write_pandas(
    df_raw,
    table_name="RAW_ENERGY",
    database="ENERGY_ML",
    schema="ML_PIPE",
    auto_create_table=False,
    overwrite=True
)

print("RAW_ENERGY loaded successfully!")
print("Rows loaded:", len(df_raw))

In [ ]:
session.write_pandas(
    df_raw,
    table_name="RAW_ENERGY",
    database="ENERGY_ML",
    schema="ML_PIPE",
    auto_create_table=False,
    overwrite=True,
    use_logical_type=True
)

print("RAW_ENERGY loaded successfully!")
print("Rows loaded:", len(df_raw))

In [ ]:
from snowflake.ml.modeling.xgboost import XGBRegressor

print("Snowpark ML XGBoost imported successfully!")

In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

In [ ]:
# ============================================================
# CELL 13 - TRAIN NATIVE XGBOOST MODEL
# ============================================================

import pandas as pd
from xgboost import XGBRegressor

print("Native XGBoost imported successfully!")


# ------------------------------------------------------------
# 1. Set Snowflake context
# ------------------------------------------------------------
session.sql("USE DATABASE ENERGY_ML").collect()
session.sql("USE SCHEMA ENERGY_ML.ML_PIPE").collect()
session.sql("USE WAREHOUSE ENERGY_ML_WH").collect()

print("Snowflake context configured successfully!")


# ------------------------------------------------------------
# 2. Load model data from Snowflake
# ------------------------------------------------------------
model_data = session.table(
    "ENERGY_ML.ML_PIPE.ENERGY_MODEL_DATA"
)

print("Model data loaded from Snowflake!")


# ------------------------------------------------------------
# 3. Define features and target
# ------------------------------------------------------------
feature_cols = [
    "HOUR",
    "DAY_OF_WEEK",
    "MONTH",
    "DAY_OF_YEAR",
    "IS_WEEKEND",
    "TEMP",
    "DWPT",
    "RHUM",
    "WDIR",
    "WSPD",
    "PRES",
    "COCO",
    "EL_PRICE"
]

label_col = "CONSUMPTION"

print("Number of features:", len(feature_cols))
print("Target:", label_col)


# ------------------------------------------------------------
# 4. Create training and validation Snowflake DataFrames
# ------------------------------------------------------------
train_data = (
    model_data
    .filter(model_data["DATASET_TYPE"] == "TRAIN")
    .select(feature_cols + [label_col])
)

validation_data = (
    model_data
    .filter(model_data["DATASET_TYPE"] == "VALIDATION")
    .select(feature_cols + [label_col])
)

print("Training and validation data prepared!")


# ------------------------------------------------------------
# 5. Convert only the selected data to Pandas
# ------------------------------------------------------------
train_pd = train_data.to_pandas()
validation_pd = validation_data.to_pandas()

print("Training rows:", len(train_pd))
print("Validation rows:", len(validation_pd))


# ------------------------------------------------------------
# 6. Separate features and target
# ------------------------------------------------------------
X_train = train_pd[feature_cols]
y_train = train_pd[label_col]

X_validation = validation_pd[feature_cols]
y_validation = validation_pd[label_col]

print("Feature and target data separated!")


# ------------------------------------------------------------
# 7. Create native XGBoost model
# ------------------------------------------------------------
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror"
)

print("XGBoost model created successfully!")


# ------------------------------------------------------------
# 8. Train the model
# ------------------------------------------------------------
print("Starting XGBoost training...")

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost model trained successfully! 🎉")

In [ ]:
# ============================================================
# CELL 14 - EVALUATE XGBOOST MODEL
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Generating validation predictions...")

# Make predictions
predictions = xgb_model.predict(X_validation)

# Calculate evaluation metrics
mae = mean_absolute_error(y_validation, predictions)
rmse = np.sqrt(mean_squared_error(y_validation, predictions))
r2 = r2_score(y_validation, predictions)

print("\n========================================")
print("MODEL EVALUATION RESULTS")
print("========================================")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("========================================")

In [ ]:
# ============================================================
# CELL 15 - INSPECT CONSUMPTION
# ============================================================

print("Training target statistics:")
print(y_train.describe())

print("\nValidation target statistics:")
print(y_validation.describe())

print("\nFirst 10 actual vs predicted values:")

comparison = pd.DataFrame({
    "ACTUAL": y_validation.values[:10],
    "PREDICTED": predictions[:10]
})

comparison

In [ ]:
# ============================================================
# CELL 16 - CHECK PREDICTIONS
# ============================================================

comparison = pd.DataFrame({
    "ACTUAL": y_validation.reset_index(drop=True),
    "PREDICTED": pd.Series(predictions)
})

print("First 20 predictions:")
display(comparison.head(20))

print("\nPrediction statistics:")
print(comparison["PREDICTED"].describe())

print("\nActual vs Predicted correlation:")
print(comparison["ACTUAL"].corr(comparison["PREDICTED"]))

In [ ]:
# ============================================================
# CELL 17 - CREATE IMPROVED TIME FEATURES
# ============================================================

import numpy as np

# Copy the original model data
improved_data = model_data.to_pandas()

# Convert TIME to datetime
improved_data["TIME"] = pd.to_datetime(improved_data["TIME"])

# Cyclical hour features
improved_data["HOUR_SIN"] = np.sin(
    2 * np.pi * improved_data["HOUR"] / 24
)

improved_data["HOUR_COS"] = np.cos(
    2 * np.pi * improved_data["HOUR"] / 24
)

# Cyclical day-of-week features
improved_data["DOW_SIN"] = np.sin(
    2 * np.pi * improved_data["DAY_OF_WEEK"] / 7
)

improved_data["DOW_COS"] = np.cos(
    2 * np.pi * improved_data["DAY_OF_WEEK"] / 7
)

# Cyclical month features
improved_data["MONTH_SIN"] = np.sin(
    2 * np.pi * improved_data["MONTH"] / 12
)

improved_data["MONTH_COS"] = np.cos(
    2 * np.pi * improved_data["MONTH"] / 12
)

print("Improved time features created!")

print("\nNew features:")
print([
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS"
])

In [ ]:
# ============================================================
# CELL 18 - PREPARE IMPROVED TRAINING DATA
# ============================================================

# Features including cyclical time features
improved_features = [
    "HOUR",
    "DAY_OF_WEEK",
    "MONTH",
    "DAY_OF_YEAR",
    "IS_WEEKEND",
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "MONTH_SIN",
    "MONTH_COS",
    "TEMP",
    "DWPT",
    "RHUM",
    "WDIR",
    "WSPD",
    "WPGT",
    "PRES",
    "COCO",
    "EL_PRICE"
]

# Split using the same chronological TRAIN/VALIDATION labels
improved_train = improved_data[
    improved_data["DATASET_TYPE"] == "TRAIN"
].copy()

improved_validation = improved_data[
    improved_data["DATASET_TYPE"] == "VALIDATION"
].copy()

# Separate features and target
X_train_improved = improved_train[improved_features]
y_train_improved = improved_train["CONSUMPTION"]

X_validation_improved = improved_validation[improved_features]
y_validation_improved = improved_validation["CONSUMPTION"]

print("Improved training data prepared!")
print("Training rows:", len(X_train_improved))
print("Validation rows:", len(X_validation_improved))
print("Number of features:", len(improved_features))

In [ ]:
# ============================================================
# CELL 19 - TRAIN IMPROVED XGBOOST MODEL
# ============================================================

from xgboost import XGBRegressor

print("Creating improved XGBoost model...")

improved_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

print("Improved XGBoost model created!")
print("Starting training...")

improved_model.fit(
    X_train_improved,
    y_train_improved
)

print("Improved XGBoost model trained successfully! 🎉")

In [ ]:
# ============================================================
# CELL 20 - EVALUATE IMPROVED MODEL
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Generating predictions...")

improved_predictions = improved_model.predict(
    X_validation_improved
)

# Calculate metrics
improved_mae = mean_absolute_error(
    y_validation_improved,
    improved_predictions
)

improved_rmse = np.sqrt(
    mean_squared_error(
        y_validation_improved,
        improved_predictions
    )
)

improved_r2 = r2_score(
    y_validation_improved,
    improved_predictions
)

print("\n========================================")
print("IMPROVED MODEL EVALUATION")
print("========================================")
print(f"MAE  : {improved_mae:.4f}")
print(f"RMSE : {improved_rmse:.4f}")
print(f"R²   : {improved_r2:.4f}")
print("========================================")

print("\nBASELINE MODEL")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("========================================")

In [ ]:
# ============================================================
# CELL 21 - CHECK RANDOM TRAIN/VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import numpy as np

print("Preparing random split...")

# Use the original 13 features
random_data = model_data.to_pandas()

X = random_data[feature_cols]
y = random_data["CONSUMPTION"]

# Random 80/20 split
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Random split created!")
print("Training rows:", len(X_train_random))
print("Validation rows:", len(X_test_random))

# Create model
random_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror"
)

print("Training random-split model...")

random_model.fit(
    X_train_random,
    y_train_random
)

# Predictions
random_predictions = random_model.predict(X_test_random)

# Metrics
random_mae = mean_absolute_error(
    y_test_random,
    random_predictions
)

random_rmse = np.sqrt(
    mean_squared_error(
        y_test_random,
        random_predictions
    )
)

random_r2 = r2_score(
    y_test_random,
    random_predictions
)

print("\n========================================")
print("RANDOM SPLIT RESULTS")
print("========================================")
print(f"MAE  : {random_mae:.4f}")
print(f"RMSE : {random_rmse:.4f}")
print(f"R²   : {random_r2:.4f}")
print("========================================")

In [ ]:
# ============================================================
# CELL 22 - TRAIN LAG-BASED XGBOOST MODEL
# ============================================================

from xgboost import XGBRegressor

print("Loading lag-based model data...")

# Load data from Snowflake
lag_model_data = session.table(
    "ENERGY_ML.ML_PIPE.ENERGY_LAG_MODEL_DATA"
).to_pandas()

print("Data loaded successfully!")
print("Total rows:", len(lag_model_data))


# ------------------------------------------------------------
# 1. Define features
# ------------------------------------------------------------
lag_features = [
    "HOUR",
    "DAY_OF_WEEK",
    "MONTH",
    "DAY_OF_YEAR",
    "IS_WEEKEND",
    "TEMP",
    "DWPT",
    "RHUM",
    "WDIR",
    "WSPD",
    "WPGT",
    "PRES",
    "COCO",
    "EL_PRICE",
    "CONSUMPTION_LAG_1H",
    "CONSUMPTION_LAG_24H",
    "CONSUMPTION_LAG_7D"
]

target = "CONSUMPTION"


# ------------------------------------------------------------
# 2. Split training and validation data
# ------------------------------------------------------------
lag_train = lag_model_data[
    lag_model_data["DATASET_TYPE"] == "TRAIN"
].copy()

lag_validation = lag_model_data[
    lag_model_data["DATASET_TYPE"] == "VALIDATION"
].copy()

X_lag_train = lag_train[lag_features]
y_lag_train = lag_train[target]

X_lag_validation = lag_validation[lag_features]
y_lag_validation = lag_validation[target]

print("Training rows:", len(X_lag_train))
print("Validation rows:", len(X_lag_validation))
print("Number of features:", len(lag_features))


# ------------------------------------------------------------
# 3. Create XGBoost model
# ------------------------------------------------------------
lag_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

print("XGBoost model created!")
print("Starting training...")


# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------
lag_model.fit(
    X_lag_train,
    y_lag_train
)

print("Lag-based XGBoost model trained successfully! 🎉")

In [ ]:
# ============================================================
# CELL 23 - EVALUATE LAG-BASED MODEL
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Generating lag-based predictions...")

# Predictions
lag_predictions = lag_model.predict(
    X_lag_validation
)

# Metrics
lag_mae = mean_absolute_error(
    y_lag_validation,
    lag_predictions
)

lag_rmse = np.sqrt(
    mean_squared_error(
        y_lag_validation,
        lag_predictions
    )
)

lag_r2 = r2_score(
    y_lag_validation,
    lag_predictions
)

print("\n========================================")
print("LAG-BASED MODEL RESULTS")
print("========================================")
print(f"MAE  : {lag_mae:.4f}")
print(f"RMSE : {lag_rmse:.4f}")
print(f"R²   : {lag_r2:.4f}")
print("========================================")

print("\nBASELINE MODEL RESULTS")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("========================================")

In [ ]:
# ============================================================
# CELL 24 - REGISTER BASELINE XGBOOST MODEL
# ============================================================

from snowflake.ml.registry import Registry

print("Creating Snowflake Model Registry...")

# Create Registry
registry = Registry(
    session=session,
    database_name="ENERGY_ML",
    schema_name="ML_PIPE"
)

print("Model Registry connected!")


# ------------------------------------------------------------
# Sample input for model signature
# ------------------------------------------------------------
sample_input = X_train.head(10)

print("Sample input prepared!")
print("Sample rows:", len(sample_input))


# ------------------------------------------------------------
# Log the trained baseline model
# ------------------------------------------------------------
print("Registering baseline XGBoost model...")

model_version = registry.log_model(
    model=xgb_model,
    model_name="ENERGY_CONSUMPTION_XGB",
    version_name="V1",
    sample_input_data=sample_input,
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    comment=(
        "Baseline XGBoost model for household energy "
        "consumption prediction. Chronological 80/20 "
        "validation used."
    )
)

print("Model registered successfully! 🎉")
print("Model: ENERGY_CONSUMPTION_XGB")
print("Version: V1")

In [ ]:
# ============================================================
# CELL 25 - VERIFY MODEL REGISTRY
# ============================================================

print("Checking registered models...")

models = registry.show_models()

models

In [ ]:
# ============================================================
# CELL 26 - VERIFY REGISTERED MODEL
# ============================================================

result = session.sql("""
    SHOW MODELS IN SCHEMA ENERGY_ML.ML_PIPE
""").collect()

for row in result:
    print(row)

In [ ]:
# ============================================================
# CELL 27 - LOAD MODEL FROM MODEL REGISTRY
# ============================================================

print("Loading registered model...")

registered_model = registry.get_model(
    "ENERGY_CONSUMPTION_XGB"
)

model_v1 = registered_model.version("V1")

print("Registered model loaded successfully!")
print("Model: ENERGY_CONSUMPTION_XGB")
print("Version: V1")

In [ ]:
# ============================================================
# CELL 28 - INFERENCE USING REGISTERED MODEL
# ============================================================

print("Running inference using registered model V1...")

# Take 10 validation samples
inference_input = X_validation.head(10).copy()

# Run prediction using the registered model
inference_output = model_v1.run(
    inference_input,
    function_name="predict"
)

print("Inference completed successfully! 🎉")

inference_output

In [ ]:
# ============================================================
# CELL 29 - REGISTER MODEL FOR WAREHOUSE INFERENCE
# ============================================================

from snowflake.ml.model import target_platform

print("Registering model for Warehouse inference...")

warehouse_model_v2 = registry.log_model(
    model=xgb_model,
    model_name="ENERGY_CONSUMPTION_XGB",
    version_name="V2",
    sample_input_data=X_train.head(10),
    target_platforms=["WAREHOUSE"],
    metrics={
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    },
    comment=(
        "Baseline XGBoost energy consumption model "
        "registered for Snowflake Warehouse inference."
    )
)

print("Model V2 registered successfully! 🎉")
print("Model: ENERGY_CONSUMPTION_XGB")
print("Version: V2")
print("Target platform: WAREHOUSE")

In [ ]:
# ============================================================
# CELL 30 - LOAD WAREHOUSE MODEL V2
# ============================================================

print("Loading Warehouse-enabled model V2...")

warehouse_model = registry.get_model(
    "ENERGY_CONSUMPTION_XGB"
).version("V2")

print("Model V2 loaded successfully! 🎉")
print("Model: ENERGY_CONSUMPTION_XGB")
print("Version: V2")
print("Target platform: WAREHOUSE")

In [ ]:
# ============================================================
# CELL 31 - RUN WAREHOUSE INFERENCE
# ============================================================

print("Running inference using registered model V2...")

# Use 10 validation samples
inference_input = X_validation.head(10).copy()

# Run prediction through the registered Snowflake model
inference_output = warehouse_model.run(
    inference_input,
    function_name="predict"
)

print("Warehouse inference completed successfully! 🎉")

inference_output

In [ ]:
print(df_raw.columns.tolist())

In [ ]:
df_raw.columns = [col.upper() for col in df_raw.columns]

print(df_raw.columns.tolist())

In [ ]:
session.write_pandas(
    df_raw,
    table_name="RAW_ENERGY",
    database="ENERGY_ML",
    schema="ML_PIPE",
    auto_create_table=False,
    overwrite=True,
    use_logical_type=True
)